# Bhagavad Gita C2 Semantic Similarity — Embeddings, Calibration, and Load

Generates pinned translation embeddings, creates a Neo4j vector index, calibrates similarity threshold via theme-coverage evaluation, and loads canonical `SIMILAR_TO` edges.

- **Spec:** `docs/superpowers/specs/2026-08-21-gita-kg-c2-semantic-similarity-design.md`
- **Prerequisites:** v1 base graph and C1 themes already loaded (run `gita_kg.ipynb` and `themes_kg.ipynb` first).
- Model: `sentence-transformers/all-mpnet-base-v2` revision `e8c3b32...`
- Deterministic + idempotent: reuses embeddings by hash, rebuilds edges with same fingerprint.

In [ ]:
import dataclasses
import hashlib
import sys
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from neo4j import GraphDatabase


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
PKG = ROOT / "gita-knowledge-graph"
sys.path.insert(0, str(PKG))

import gita_embeddings as ge
import gita_kg as gk

## 1. Config & connect

In [ ]:
load_dotenv(PKG / ".env", override=True)
cfg = gk.load_config()


def new_driver():
    return GraphDatabase.driver(cfg.uri, auth=(cfg.user, cfg.password))


with new_driver() as driver:
    driver.verify_connectivity()
print("connected:", cfg.uri, "->", cfg.database)

emb_config = gk.EmbeddingConfig()
print(f"embedding config: {emb_config.model_id} @ {emb_config.revision[:7]}...")
print(f"dimensions={emb_config.dimensions}, top_k={emb_config.top_k}, threshold={emb_config.threshold}")

## 2. Read verses with existing embeddings and themes

In [ ]:
READ_QUERY = """
MATCH (v:Verse)
OPTIONAL MATCH (v)-[:MENTIONS_THEME]->(th:Theme)
RETURN v.id AS id, v.chapter AS chapter, v.verse AS verse,
       v.translation AS translation, v.embedding AS embedding,
       v.embedding_model AS embedding_model,
       v.embedding_revision AS embedding_revision,
       v.embedding_dimension AS embedding_dimension,
       v.embedding_input_sha256 AS embedding_input_sha256,
       collect(th.name) AS themes
ORDER BY v.chapter, v.verse
"""

with new_driver() as driver, driver.session(database=cfg.database) as session:
    rows = [dict(r) for r in session.run(READ_QUERY)]

print(f"read {len(rows)} verses")
print(f"sample: {rows[0]['id']} — {rows[0]['translation'][:60]}...")

## 3. Compute embeddings with reuse-by-hash

In [ ]:
def sha256(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


stale_rows = []
for row in rows:
    current_hash = sha256(row["translation"])
    is_stale = (
        row["embedding"] is None
        or row["embedding_model"] != emb_config.model_id
        or row["embedding_revision"] != emb_config.revision
        or row["embedding_dimension"] != emb_config.dimensions
        or row["embedding_input_sha256"] != current_hash
    )
    if is_stale:
        stale_rows.append({"id": row["id"], "translation": row["translation"], "input_sha256": current_hash})
    else:
        row["final_embedding"] = row["embedding"]

print(f"stale embeddings: {len(stale_rows)} / {len(rows)}")

if stale_rows:
    print("loading model...")
    model = ge.load_embedding_model(emb_config)
    print("encoding stale translations...")
    stale_matrix = ge.encode_translations(
        model, [r["translation"] for r in stale_rows], emb_config.dimensions
    )
    stale_dict = {stale_rows[i]["id"]: stale_matrix[i].tolist() for i in range(len(stale_rows))}
    for row in rows:
        if row["id"] in stale_dict:
            row["final_embedding"] = stale_dict[row["id"]]
    print(f"encoded {len(stale_rows)} fresh embeddings")

    persist_rows = [
        {"id": sr["id"], "embedding": stale_dict[sr["id"]], "input_sha256": sr["input_sha256"]}
        for sr in stale_rows
    ]
    ops = gk.embedding_ops(persist_rows, emb_config)
    with new_driver() as driver, driver.session(database=cfg.database) as session:
        for cypher, params in ops:
            session.run(cypher, **params)
    print("persisted embeddings to Neo4j")
else:
    print("all embeddings up to date, reusing")

## 4. Assemble and validate embedding matrix

In [ ]:
ids = [r["id"] for r in rows]
embeddings = np.array([r["final_embedding"] for r in rows], dtype=np.float32)
print(f"embedding matrix shape: {embeddings.shape}")
assert embeddings.shape == (701, 768), f"expected (701, 768), got {embeddings.shape}"

norms = np.linalg.norm(embeddings, axis=1)
np.testing.assert_allclose(norms, 1.0, atol=1e-5)
print(f"all {len(norms)} embeddings are unit-normalized (max deviation: {abs(norms - 1.0).max():.2e})")

## 5. Create vector index and verify online

In [ ]:
ops = gk.vector_index_ops(emb_config)
with new_driver() as driver, driver.session(database=cfg.database) as session:
    for cypher, params in ops:
        session.run(cypher, **params)
print("vector index creation command issued")

VERIFY_QUERY = """
SHOW VECTOR INDEXES
YIELD name, state, labelsOrTypes, properties, options
WHERE name = 'verse_translation_embeddings'
RETURN name, state, labelsOrTypes, properties, options
"""

max_retries = 30
for attempt in range(1, max_retries + 1):
    with new_driver() as driver, driver.session(database=cfg.database) as session:
        result = session.run(VERIFY_QUERY).single()
    if result and result["state"] == "ONLINE":
        print(f"vector index ONLINE after {attempt} checks")
        print(f"  name: {result['name']}")
        print(f"  labels: {result['labelsOrTypes']}")
        print(f"  properties: {result['properties']}")
        print(f"  options: {result['options']}")
        break
    time.sleep(0.5)
else:
    raise TimeoutError(f"vector index not ONLINE after {max_retries} retries")

options = result["options"]
assert result["labelsOrTypes"] == ["Verse"], "index label mismatch"
assert result["properties"] == ["embedding"], "index property mismatch"
assert options["indexConfig"]["vector.dimensions"] == 768, "dimensions mismatch"
assert options["indexConfig"]["vector.similarity_function"] == "cosine", "similarity function mismatch"
print("✓ index configured: Verse.embedding, 768 dimensions, cosine similarity")

## 6. Compute exact cosine similarity matrix

In [ ]:
similarity_matrix = embeddings @ embeddings.T
print(f"similarity matrix shape: {similarity_matrix.shape}")
print(f"symmetric: {np.allclose(similarity_matrix, similarity_matrix.T, atol=1e-8)}")
print(f"diagonal range: [{similarity_matrix.diagonal().min():.6f}, {similarity_matrix.diagonal().max():.6f}]")

## 7. Build pairs at threshold -1.0 and print score quantiles

In [ ]:
all_pairs = gk.build_similarity_pairs(ids, similarity_matrix, emb_config.top_k, threshold=-1.0)
print(f"total directed top-5 pairs (threshold=-1.0): {len(all_pairs)}")

scores = [p.score for p in all_pairs]
quantiles = gk.similarity_score_quantiles(scores)
print("\nsimilarity score quantiles:")
for key, val in quantiles.items():
    print(f"  {key:>4s}: {val:.4f}")

## 8. Build auxiliary data and evaluate candidate thresholds

In [ ]:
chapters = {r["id"]: r["chapter"] for r in rows}
themes_by_id = {r["id"]: set(r["themes"]) - {None} for r in rows}

all_theme_names = set(gk.THEMES.keys())
stats_list = gk.evaluate_thresholds(
    ids, similarity_matrix, emb_config.top_k, gk.CANDIDATE_THRESHOLDS, chapters, themes_by_id
)

print("\nthreshold evaluation:")
print(f"{'threshold':>9s} {'coverage':>8s} {'pairs':>6s} {'mutual':>6s} {'cross_ch_themes':>15s}")
for st in stats_list:
    print(
        f"{st.threshold:>9.2f} {st.covered_fraction:>8.2%} {st.pair_count:>6d} {st.mutual_count:>6d} "
        f"{len(st.cross_chapter_themes):>15d}"
    )

## 9. Select threshold and build final config

In [ ]:
selected_threshold = gk.select_similarity_threshold(stats_list, all_theme_names)
final_config = dataclasses.replace(emb_config, threshold=selected_threshold)
print(f"\nselected threshold: {selected_threshold:.2f}")
print(f"final config: model={final_config.model_id[:30]}..., threshold={final_config.threshold}")

selected_stats = [st for st in stats_list if st.threshold == selected_threshold][0]
print(f"  coverage: {selected_stats.covered_fraction:.2%}")
print(f"  pairs: {selected_stats.pair_count}")
print(f"  mutual: {selected_stats.mutual_count}")
print(f"  cross-chapter themes: {len(selected_stats.cross_chapter_themes)} / {len(all_theme_names)}")

if len(selected_stats.cross_chapter_themes) < len(all_theme_names):
    missing = all_theme_names - selected_stats.cross_chapter_themes
    print(f"  WARNING: missing themes: {sorted(missing)}")

## 10. Sample quality gate

Run this cell, inspect the printed neighbours, lowest-scoring pairs, and cross-chapter samples, then change `QUALITY_APPROVED` to `True` and rerun the cell. The load cell refuses to mutate Neo4j until approval is explicit.

In [ ]:
final_pairs = gk.build_similarity_pairs(ids, similarity_matrix, emb_config.top_k, selected_threshold)
print(f"\nfinal pair count at threshold {selected_threshold}: {len(final_pairs)}")

pairs_247 = [p for p in final_pairs if "2.47" in (p.a_id, p.b_id)]
print(f"\nNeighbours of 2.47 (n={len(pairs_247)}):")
for p in pairs_247[:5]:
    other = p.b_id if p.a_id == "2.47" else p.a_id
    print(f"  {other}: score={p.score:.4f}, mutual={p.mutual}")

sorted_pairs = sorted(final_pairs, key=lambda p: p.score)
print(f"\nTen lowest-scoring pairs at threshold {selected_threshold}:")
for p in sorted_pairs[:10]:
    print(f"  ({p.a_id}, {p.b_id}): {p.score:.4f}")

rng = np.random.default_rng(42)
cross_chapter = [p for p in final_pairs if chapters[p.a_id] != chapters[p.b_id]]
sample_indices = rng.choice(len(cross_chapter), size=min(10, len(cross_chapter)), replace=False)
print(f"\nTen random cross-chapter pairs (seed=42, n={len(cross_chapter)} available):")
for idx in sample_indices:
    p = cross_chapter[idx]
    print(f"  ({p.a_id}, {p.b_id}): {p.score:.4f}, chapters=({chapters[p.a_id]}, {chapters[p.b_id]})")

QUALITY_APPROVED = False
print("\nHALT: inspect the samples above. Set QUALITY_APPROVED = True and rerun this cell before loading edges.")

## 11. Clear stale edges and load SIMILAR_TO relationships

In [ ]:
if not QUALITY_APPROVED:
    raise RuntimeError("quality gate not approved; inspect samples and set QUALITY_APPROVED = True")


def run_ops(ops):
    with new_driver() as driver, driver.session(database=cfg.database) as session:
        for cypher, params in ops:
            session.run(cypher, **params)


print("clearing stale SIMILAR_TO edges...")
run_ops(gk.clear_similarity_ops())
print("loading new SIMILAR_TO edges...")
run_ops(gk.similarity_ops(final_pairs, final_config))
print(f"loaded {len(final_pairs)} SIMILAR_TO edges")

## 12. Evaluation queries

In [ ]:
def query_one(cypher, params=None):
    with new_driver() as driver, driver.session(database=cfg.database) as session:
        return session.run(cypher, params or {}).single()[0]


def query_all(cypher, params=None):
    with new_driver() as driver, driver.session(database=cfg.database) as session:
        return [dict(r) for r in session.run(cypher, params or {})]


total_edges = query_one("MATCH ()-[r:SIMILAR_TO]->() RETURN count(r)")
print(f"\ntotal canonical SIMILAR_TO pairs: {total_edges}")

incident_verses = query_one("MATCH (v:Verse)-[:SIMILAR_TO]-() RETURN count(DISTINCT v)")
print(f"verses with >=1 edge: {incident_verses} / 701 ({incident_verses / 701:.2%})")

mutual_edges = query_one("MATCH ()-[r:SIMILAR_TO]->() WHERE r.mutual = true RETURN count(r)")
print(f"mutual edges: {mutual_edges} / {total_edges} ({mutual_edges / total_edges:.2%})")

same_chapter = query_one(
    "MATCH (a:Verse)-[r:SIMILAR_TO]->(b:Verse) WHERE a.chapter = b.chapter RETURN count(r)"
)
cross_chapter_edges = total_edges - same_chapter
print(f"same-chapter edges: {same_chapter} ({same_chapter / total_edges:.2%})")
print(f"cross-chapter edges: {cross_chapter_edges} ({cross_chapter_edges / total_edges:.2%})")

shared_theme = query_one("""
MATCH (a:Verse)-[:SIMILAR_TO]->(b:Verse)
WHERE EXISTS { MATCH (a)-[:MENTIONS_THEME]->(th:Theme)<-[:MENTIONS_THEME]-(b) }
RETURN count(*)
""")
print(f"edges with shared theme: {shared_theme} / {total_edges} ({shared_theme / total_edges:.2%})")

cross_themes = query_all("""
MATCH (a:Verse)-[:SIMILAR_TO]->(b:Verse)
WHERE a.chapter <> b.chapter
MATCH (a)-[:MENTIONS_THEME]->(th:Theme)<-[:MENTIONS_THEME]-(b)
RETURN th.name AS theme, count(*) AS edge_count
ORDER BY edge_count DESC
""")
print(f"\ncross-chapter themes (n={len(cross_themes)}):")
for row in cross_themes:
    print(f"  {row['theme']:>20s}: {row['edge_count']:>4d}")

isolated = query_all("""
MATCH (v:Verse)
WHERE NOT (v)-[:SIMILAR_TO]-()
RETURN v.id AS id
ORDER BY v.chapter, v.verse
""")
print(f"\nisolated verses (no SIMILAR_TO edges): {len(isolated)}")
if isolated:
    print(f"  sample: {[r['id'] for r in isolated[:10]]}")

edge_scores = query_all("MATCH ()-[r:SIMILAR_TO]->() RETURN r.score AS score")
edge_score_list = [r["score"] for r in edge_scores]
edge_quantiles = gk.similarity_score_quantiles(edge_score_list)
print("\nedge score quantiles:")
for key, val in edge_quantiles.items():
    print(f"  {key:>4s}: {val:.4f}")

neighbours_247 = query_all("""
MATCH (v:Verse {id: $verse_id})-[r:SIMILAR_TO]-(n:Verse)
RETURN n.id AS id, r.score AS score, r.mutual AS mutual
ORDER BY r.score DESC
""", {"verse_id": "2.47"})
print(f"\nrepresentative neighbours for 2.47 (n={len(neighbours_247)}):")
for row in neighbours_247[:5]:
    print(f"  {row['id']}: score={row['score']:.4f}, mutual={row['mutual']}")

for theme in ["bhakti", "moksha", "brahman"]:
    sample_verse = query_all("""
    MATCH (v:Verse)-[:MENTIONS_THEME]->(:Theme {name: $theme})
    RETURN v.id AS id
    ORDER BY v.chapter, v.verse
    LIMIT 1
    """, {"theme": theme})
    if sample_verse:
        vid = sample_verse[0]["id"]
        neighbours = query_all("""
        MATCH (v:Verse {id: $verse_id})-[r:SIMILAR_TO]-(n:Verse)
        RETURN n.id AS id, r.score AS score
        ORDER BY r.score DESC
        LIMIT 3
        """, {"verse_id": vid})
        print(f"\nrepresentative neighbours for {theme} verse {vid} (n={len(neighbours)}):")
        for row in neighbours:
            print(f"  {row['id']}: {row['score']:.4f}")

## 13. Vector query sample for verse 2.47

In [ ]:
verse_247_embedding = [r["final_embedding"] for r in rows if r["id"] == "2.47"][0]

vector_results = query_all("""
CALL db.index.vector.queryNodes('verse_translation_embeddings', 6, $embedding)
YIELD node, score
RETURN node.id AS id, score
""", {"embedding": verse_247_embedding})

print("\nvector query for 2.47 (top 6 including self):")
for row in vector_results:
    print(f"  {row['id']}: {row['score']:.4f}")

## 14. Idempotency check

In [ ]:
def fingerprint_edges():
    edges = query_all("""
    MATCH (a:Verse)-[r:SIMILAR_TO]->(b:Verse)
    RETURN a.id AS a_id, b.id AS b_id, r.score AS score, r.mutual AS mutual,
           r.rank_a AS rank_a, r.rank_b AS rank_b, r.model AS model,
           r.revision AS revision, r.top_k AS top_k, r.threshold AS threshold
    ORDER BY a.chapter, a.verse, b.chapter, b.verse
    """)
    canonical = [
        f"{e['a_id']}|{e['b_id']}|{e['score']:.6f}|{e['mutual']}|{e['rank_a']}|{e['rank_b']}|"
        f"{e['model']}|{e['revision']}|{e['top_k']}|{e['threshold']:.2f}"
        for e in edges
    ]
    digest = hashlib.sha256("\n".join(canonical).encode()).hexdigest()
    return len(edges), digest


count_1, fp_1 = fingerprint_edges()
print(f"\nfingerprint run 1: count={count_1}, sha256={fp_1[:16]}...")

# Rebuild
print("rebuilding edges...")
run_ops(gk.clear_similarity_ops())
run_ops(gk.similarity_ops(final_pairs, final_config))
count_2, fp_2 = fingerprint_edges()
print(f"fingerprint run 2: count={count_2}, sha256={fp_2[:16]}...")

assert count_1 == count_2, f"edge count mismatch: {count_1} vs {count_2}"
assert fp_1 == fp_2, f"fingerprint mismatch: {fp_1[:16]}... vs {fp_2[:16]}..."
print("✓ idempotency verified: count and fingerprint stable across rebuilds")

## 15. Close driver

In [ ]:
print("all Neo4j drivers were opened in context managers and closed after each operation")